# Sprint C-Training: fit BMA priors + isotonic calibrators + conformal residuals on Table 3-1

This notebook is the **reproducible end-to-end training run** for the seven Table 3-1 SEP events. It produces the four trained artifacts that the kill-gate hold-out evaluation will consume:

1. `bma_priors_table_3_1.npz` -- per-event BMA weight vectors
2. `isotonic_calibrators_stratified.npz` -- one isotonic calibrator per Kp severity stratum
3. `conformal_residuals_split.npz` -- marginal split-conformal residuals
4. `conformal_residuals_mondrian.npz` -- per-stratum Mondrian conformal residuals

Each artifact ships with a sibling `.provenance.json` validating against `helios-provenance-spec` v0.1 (`HeliosTransformationRecord`).

**Critical constraint reminders** (see `helios-program/specs/2026-05-17-Sprint-C-Training-spec.md`):

- Does **not** run hold-out evaluation -- that's gated on OSF pre-registration filing.
- Hyperparameters are **locked** per the OSF pre-registration template.
- Synthetic-proxy fallbacks are used for upstream sources that don't cover the event window (especially pre-2018 ISWA Scoreboard data); each fallback is recorded in the per-event `data_gaps` map.

## Setup

In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import numpy as np

from helios_fusion.training import (
    TRAINING_EVENTS,
    load_all_training_events,
    run_full_training,
)
from helios_fusion.training.pipeline import persist_artifacts

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
print("Training events:")
for ev in TRAINING_EVENTS:
    print(f"  - {ev.event_id:25s} {ev.onset.isoformat()}  {ev.notes}")

## 1. Load Table 3-1 training events

We invoke the connector-driven loader with `use_real_data=True`. Each event-window pull attempts live ISWA / SWPC fetches; when an upstream source doesn't cover the window, a documented synthetic-proxy stream is substituted and the deferral is recorded in `data_gaps`.

Set `use_real_data=False` to run the notebook purely against synthetic streams (faster; matches the kill-gate's pre-flight check).

In [ ]:
USE_REAL_DATA = True  # set to False for a fully-offline rerun
CADENCE_HOURS = 1.0

frames = load_all_training_events(
    use_real_data=USE_REAL_DATA,
    cadence_hours=CADENCE_HOURS,
)

for frame in frames:
    iswa_rows = int((frame.records["source"] == "iswa").sum()) if not frame.records.empty else 0
    synth_rows = int((frame.records["source"] == "synthetic_proxy").sum()) if not frame.records.empty else 0
    print(
        f"{frame.event.event_id:25s} n_rows={len(frame.records):5d}  "
        f"iswa={iswa_rows:5d} synthetic={synth_rows:5d}  "
        f"gaps={len(frame.data_gaps)}"
    )

## 2. Fit BMA + isotonic + conformal

The `run_full_training` orchestrator calls the four fits in sequence. Each fit is exposed as a standalone function in `helios_fusion.training` for callers that need finer control.

In [ ]:
artifacts = run_full_training(frames=frames)

print("BMA weights (top model per event):")
for event_id, weights in artifacts.bma_priors.items():
    top_model = max(weights, key=lambda k: weights[k])
    print(f"  {event_id:25s}  top={top_model:30s}  w={weights[top_model]:.4f}")

print(f"\nCalibrator per-stratum counts: {artifacts.calibrator.sample_counts}")
print(f"Split conformal n_calibration: {artifacts.split_conformal.n_calibration}")
print(f"Mondrian per-stratum counts:   {artifacts.mondrian_conformal.per_stratum_counts()}")

## 3. Quick diagnostic: residual quantile at α=0.1

The locked kill-gate alpha is 0.1 (90% prediction intervals). Compute the split-conformal half-width and the per-stratum Mondrian half-widths.

In [ ]:
probe = np.array([0.5])
split_int = artifacts.split_conformal.predict_interval(probe, alpha=0.1)
split_halfwidth = float((split_int[0, 1] - split_int[0, 0]) / 2.0)
print(f"Split conformal half-width (α=0.1): {split_halfwidth:.4f}")

for stratum in ("quiet", "moderate", "extreme"):
    interval = artifacts.mondrian_conformal.predict_interval(probe, [stratum], alpha=0.1)
    hw = float((interval[0, 1] - interval[0, 0]) / 2.0)
    print(f"Mondrian {stratum:9s} half-width (α=0.1): {hw:.4f}")

## 4. Persist artifacts

Writes four `.npz` archives + four `.provenance.json` files + the `manifest.json` to the `helios-fusion-internal/weights/` private companion repo. The OSF pre-registration URL is left as `null` until the operator files.

In [ ]:
WEIGHTS_DIR = Path.home() / "577i-Projects" / "helios-fusion-internal" / "weights"
REPO_ROOT = Path.cwd().parent  # notebook lives in <repo>/notebooks/

written = persist_artifacts(
    artifacts,
    weights_dir=WEIGHTS_DIR,
    repo_root=REPO_ROOT,
    osf_preregistration_url=None,
)
for name, path in written.items():
    print(f"  {name:25s} -> {path}")

manifest = json.loads((WEIGHTS_DIR / "manifest.json").read_text())
print(f"\nManifest training_run_date_utc: {manifest['training_run_date_utc']}")
print(f"Manifest git_sha:               {manifest['fusion_engine_git_sha']}")
print(f"OSF preregistration URL:        {manifest['osf_preregistration_url']}")

## 5. Synthetic-data sanity test (the kill-gate's pre-flight check)

Run the full pipeline once more in synthetic-only mode and check that all four artifacts are produced. This is the test that the kill-gate runner replays before any hold-out evaluation.

In [ ]:
synth_artifacts = run_full_training(use_real_data=False, cadence_hours=4.0)
assert len(synth_artifacts.bma_priors) == 7, "need seven event priors"
assert synth_artifacts.calibrator.fitted
assert synth_artifacts.split_conformal.fitted
assert synth_artifacts.mondrian_conformal.fitted
print("synthetic-data sanity test: PASS")